In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.environ.get("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DCB520B830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DCB63586E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
model.invoke([
    HumanMessage(content="Hello, My name is Kajal")
])

AIMessage(content='Hello Kajal! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hello, My name is Kajal". They just introduced themselves. The system instructions: The assistant is an AI assistant. We need to respond in a friendly manner. No conflict. Just respond.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 78, 'total_tokens': 144, 'completion_time': 0.067045488, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.003716766, 'prompt_tokens_details': None, 'queue_time': 0.274921143, 'total_time': 0.070762254}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05716-37ae-7fe2-a3c7-1e308a9d3d38-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 66, 'total_tokens': 144, 'output_token_details': {'reasoning': 44}})

In [4]:
model.invoke([
    HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer"),
    AIMessage(content="'Hello Kajal! 👋 How’s your day going? Anything in particular you’d like to chat about or need help with?'"),
    HumanMessage(content="Hey, What's my name and What do i do for a living?"),
])

AIMessage(content='You’re Kajal, and you work as a **Blockchain Developer**. 🚀', additional_kwargs={'reasoning_content': 'User says "Hello, My name is Kajal and I am a Blockchain Developer". So name is Kajal, occupation blockchain developer. The user asks "Hey, What\'s my name and What do i do for a living?" So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 135, 'total_tokens': 211, 'completion_time': 0.108227489, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.006775588, 'prompt_tokens_details': None, 'queue_time': 0.313291521, 'total_time': 0.115003077}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05716-39d2-70d0-b31f-560ba330edd6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 76, 'total_tokens': 211, 'output_token_de

## Message History

We can use a message history class to wrap our model and make it stateful.
This wll keep track of inputs and outputs of the model, and store them in some datastore.
Future interactions will then load messages and pass them into the chain as part of the input.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

C:\Users\kajal\AppData\Local\Temp\ipykernel_15652\3355908701.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer")],
     config=config
)

In [8]:
response.content

'Hi Kajal! 👋 It’s great to meet a fellow blockchain developer. How can I help you today? Are you working on a particular project or looking for insights on a specific topic?'

In [9]:
with_message_history.invoke([HumanMessage(content="Hey, What's my name and What do i do for a living?")], config=config).content

'You’re Kajal, and you work as a **Blockchain Developer**.'

In [10]:
## change the config ===> session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hello, What is my name")],
     config=config1
)
response.content

'I don’t have that information. What’s your name?'

## Prompt Templates

Prompt Templates help to turn raw user information into a format that the LLM can work with.
In this case, the raw user input is just a message, which we are passing to the LLM.
Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages 

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant that answers questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain=prompt | model

In [12]:
chain.invoke({"messages": [HumanMessage(content="Hello, My name is Kajal"),]})

AIMessage(content='Hello Kajal! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hello, My name is Kajal". Likely greeting. We respond politely. No request. So just greet back.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 96, 'total_tokens': 147, 'completion_time': 0.069388318, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.038165977, 'prompt_tokens_details': None, 'queue_time': 0.317107109, 'total_time': 0.107554295}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1c0f5282a2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05716-45ea-77b0-9d38-d1a505ae9d8a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 51, 'total_tokens': 147, 'output_token_details': {'reasoning': 29}})

In [13]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer")],
     config=config  )       

response.content 

'Hello Kajal! 👋 It’s great to meet a fellow blockchain enthusiast. How can I help you today? Whether it’s a technical question, project idea, or anything else, I’m here to assist!'

In [15]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant that answers questions to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain=prompt | model

In [16]:
response = chain.invoke({
    "messages": [HumanMessage(content="Hello, My name is Kajal and I am a Blockchain Developer")],
    "language": "Hindi"
})

In [17]:
response.content

'नमस्ते काजल जी! आपका स्वागत है। मैं आपकी कैसे सहायता कर सकता/सकती हूँ? यदि आपको ब्लॉकचेन से जुड़ी किसी विषय पर चर्चा करनी है, कोडिंग में मदद चाहिए, या किसी प्रोजेक्ट के लिए सलाह चाहते हैं, तो बेझिझक बताइए!'

Let's now wrap this more complicated chain in a message history class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history

In [18]:
with_message_history=RunnableWithMessageHistory(chain, get_session_history,input_messages_key="messages")

c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages' : [HumanMessage(content="Hello, My name is Kajal")],"language": "Hindi"},
     config=config  )

response.content

'नमस्ते काजल! आपसे मिलकर खुशी हुई। मैं आपकी किस प्रकार मदद कर सकता/सकती हूँ?'

In [20]:
response=with_message_history.invoke(
    {'messages' : [HumanMessage(content="What's my name")],"language": "Spanish"},
     config=config  )

response.content

'Tu nombre es Kajal.'

In [21]:
response=with_message_history.invoke(
    {'messages' : [HumanMessage(content="What's my name")],"language": "Hindi"},
     config=config  )

response.content

'आपका नाम काजल है।'

## Managing the converstaion history

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

In [22]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=70, 
    strategy="last",
    token_counter=model,
    allow_partial=False,
    start_on="human"
    )

messages = [
    SystemMessage(content="You are a helpful assistant that answers questions to the best of your ability"),
    HumanMessage(content="Hello, I am bob"),
    AIMessage(content="Hello"),
    HumanMessage(content="I like vanila ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="what is 3+8"),
    AIMessage(content="11"),
    HumanMessage(content="thanks"),
    AIMessage(content="You're welcome!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="Yes, I am!"),
]

trimmer.invoke(messages)

c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\kajal\OneDrive\Dokumen\Langchain\venev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[HumanMessage(content='Hello, I am bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanila ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is 3+8', additional_kwargs={}, response_metadata={}),
 AIMessage(content='11', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes, I am!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [24]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer) 
    | prompt 
    | model
)

response = chain.invoke({"messages": messages + [HumanMessage(content="what ice ceream flavor do i like?")],
                         "language": "English"})

response.content

'You mentioned you like vanilla ice cream, so that’s the flavor you’re into!'